In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

# 環境変数の取得
load_dotenv()

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 1. Excelファイルを読み込む
df = pd.read_excel('サンプルデータ.xlsx', sheet_name='売上データ')
# データフレームを表示して確認
df.head()

,カテゴリー,商品コード,商品名,売上日,単価,数量,原価
0,食品,1001,りんご,2023-01-01,200,50,120
1,食品,1002,バナナ,2023-01-01,150,100,80
2,食品,1003,牛乳,2023-01-02,180,80,100
3,衣服,2001,Tシャツ,2023-01-02,1500,20,800
4,衣服,2002,ジーンズ,2023-01-03,5000,10,2500


In [3]:
# 2. データをLLM用にテキスト形式に変換
# データフレーム全体を文字列に変換
sales_data_text = df.astype(str)
prompt_text = f"売上データ:\n{sales_data_text}\nこの売上データの傾向を分析してください。"
# 表示して確認
print(prompt_text)

売上データ:
    カテゴリー 商品コード      商品名         売上日    単価   数量    原価
0      食品  1001      りんご  2023-01-01   200   50   120
1      食品  1002      バナナ  2023-01-01   150  100    80
2      食品  1003       牛乳  2023-01-02   180   80   100
3      衣服  2001     Tシャツ  2023-01-02  1500   20   800
4      衣服  2002     ジーンズ  2023-01-03  5000   10  2500
..    ...   ...      ...         ...   ...  ...   ...
235    衣服  2077   レインパンツ  2023-04-28  2000   18  1000
236    食品  1085      ザクロ  2023-04-29   600   40   300
237   日用品  3077    バスブラシ  2023-04-29   400   60   200
238    衣服  2078  レインシューズ  2023-04-30  2500   15  1250
239    食品  1086    ココナッツ  2023-04-30   300   80   150

[240 rows x 7 columns]
この売上データの傾向を分析してください。


In [4]:
# 3. OpenAI APIの呼び出し

# 役割を設定
role = "あなたはマーケティング分野に精通したデータサイエンティストです。企業の成長をサポートするために、効果的なインサイトを提供します。"

# APIへリクエスト
response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": role},
        {"role": "user", "content": prompt_text},
    ],
)

# LLMからの回答を表示
print(response.choices[0].message.content.strip())

売上データを分析することで、いくつかの傾向やインサイトを導き出すことができます。以下に、売上データを分析する際のステップと主な観察結果を示します。

### 1. 総売上の把握
まず、全体の売上を計算し、期間内の売上トレンドを把握します。この場合、各商品の「単価」と「数量」を掛け合わせて総売上を求め、それを期間で集計します。

### 2. カテゴリー別の売上分析
各カテゴリー（食品、衣服、日用品）ごとの売上を分析します。これにより、どのカテゴリが最も売上を上げているか、またはどのカテゴリが売上を伸ばしているかを把握できます。

### 3. 商品別の売上分析
具体的な商品名ごとの売上を分析し、BESTセラー商品や売上が低迷している商品を見つけます。これにより、商品のラインナップを調整するためのインサイトを得られます。

### 4. 時間的なトレンド
売上データを日付ごとに集計し、売上が増加または減少している時期を把握します。季節的なトレンドや特定のプロモーション期間を特定できます。たとえば、特定の月に特売を行った際の売上の反応を観察します。

### 5. 粗利の計算
売上から原価を差し引くことで粗利を計算し、各商品やカテゴリーの利益率を分析します。どの製品が最も利益を生んでいるのか、逆に原価が高く利益が少ない商品を特定します。

### 主な傾向と示唆

1. **カテゴリーのパフォーマンス**:
   - 食品カテゴリーは通常、必要不可欠な商品を含んでいるため、常に安定した売上を叩き出すことが予想されます。
   - 衣服カテゴリーは季節や流行に左右されやすく、特定の期間（春の新作、年末セールなど）での売上が特に高くなる可能性があります。

2. **ベストセラー商品の特定**:
   - 売上が高い商品は今後のプロモーションやマーケティング戦略の中心に据えるべきです。逆に、売上の低い商品はオフラインやオンラインでの在庫見直しを検討する価値があります。

3. **シーズン要因**:
   - 売上データから、特定のシーズンに対する需要の変化を確認することが重要です。例えば、冬場には暖かい衣服の需要が高まることが期待されます。

4. **価格設定戦略**:
   - 商品の価格は需要に大きな影響を与えるため、競合の価格や市場の需要に応じた価格戦略を検討する